# M5 Demand Forecasting - Exploratory Data Analysis

This notebook performs exploratory analysis of the M5 Walmart sales dataset.

**Author:** Heer Patel  
**Dataset:** M5 Forecasting - Accuracy

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.loader import M5DataLoader
from data.cleaner import DataCleaner
from data.transformer import DataTransformer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load M5 dataset
loader = M5DataLoader('../data/raw')
sales_df, calendar_df, prices_df = loader.load_all()

print(f"Sales data shape: {sales_df.shape}")
print(f"Calendar data shape: {calendar_df.shape}")
print(f"Prices data shape: {prices_df.shape}")

## 2. Inspect Data Structure

In [ ]:
# Sales data structure
print("Sales Data - First 5 rows:")
sales_df.head()

In [ ]:
# Calendar features
print("Calendar Data - First 10 rows:")
calendar_df.head(10)

In [ ]:
# Price data
print("Prices Data - Sample:")
prices_df.head()

## 3. Data Hierarchy

In [ ]:
# Analyze hierarchy
print("Data Hierarchy:")
print(f"Number of states: {sales_df['state_id'].nunique()}")
print(f"Number of stores: {sales_df['store_id'].nunique()}")
print(f"Number of categories: {sales_df['cat_id'].nunique()}")
print(f"Number of departments: {sales_df['dept_id'].nunique()}")
print(f"Number of items (SKUs): {sales_df['item_id'].nunique()}")
print(f"Total time series: {len(sales_df)}")

In [ ]:
# Store-Category combinations
print("\nStore × Category Analysis:")
store_cat_combos = sales_df.groupby(['store_id', 'cat_id']).size().reset_index(name='n_items')
print(f"Total Store-Category combinations: {len(store_cat_combos)}")
print("\nBreakdown:")
print(store_cat_combos)

## 4. Transform to Long Format

In [ ]:
# Transform to long format
transformer = DataTransformer()
df_long = transformer.transform_to_long_format(sales_df, calendar_df)

print(f"Long format shape: {df_long.shape}")
df_long.head()

## 5. Time Series Analysis

In [ ]:
# Aggregate to store-category level
df_agg = transformer.aggregate_to_store_category(df_long)

print(f"Aggregated data shape: {df_agg.shape}")
df_agg.head()

In [ ]:
# Select one store-category for visualization
sample_store = 'CA_1'
sample_category = 'FOODS'

sample_ts = df_agg[(df_agg['store_id'] == sample_store) & 
                   (df_agg['cat_id'] == sample_category)].copy()
sample_ts = sample_ts.sort_values('date')

print(f"Sample time series: {sample_store} - {sample_category}")
print(f"Date range: {sample_ts['date'].min()} to {sample_ts['date'].max()}")
print(f"Number of days: {len(sample_ts)}")

In [ ]:
# Plot sample time series
plt.figure(figsize=(16, 6))
plt.plot(sample_ts['date'], sample_ts['units_sold'], linewidth=1.5, color='#2E86AB')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Units Sold', fontsize=12)
plt.title(f'Daily Sales - {sample_store} - {sample_category}', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Sales Distribution Analysis

In [ ]:
# Overall sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_agg['units_sold'], bins=100, color='#2E86AB', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Units Sold', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Daily Sales', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df_agg['units_sold'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#A23B72', alpha=0.7))
axes[1].set_ylabel('Units Sold', fontsize=12)
axes[1].set_title('Sales Box Plot', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sales Statistics:")
print(df_agg['units_sold'].describe())

## 7. Seasonality Analysis

In [ ]:
# Add time features
df_agg = transformer.add_time_features(df_agg)

# Day of week analysis
dow_avg = df_agg.groupby('day_of_week')['units_sold'].mean()

plt.figure(figsize=(10, 5))
plt.bar(range(7), dow_avg.values, color='#2E86AB', edgecolor='black', alpha=0.8)
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Average Units Sold', fontsize=12)
plt.title('Weekly Seasonality Pattern', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Weekend vs Weekday
weekend_avg = df_agg.groupby('is_weekend')['units_sold'].mean()

plt.figure(figsize=(8, 5))
plt.bar(['Weekday', 'Weekend'], weekend_avg.values, color=['#2E86AB', '#A23B72'], 
        edgecolor='black', alpha=0.8)
plt.ylabel('Average Units Sold', fontsize=12)
plt.title('Weekday vs Weekend Sales', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nWeekday average: {weekend_avg[0]:.2f}")
print(f"Weekend average: {weekend_avg[1]:.2f}")
print(f"Difference: {((weekend_avg[1] - weekend_avg[0]) / weekend_avg[0] * 100):.1f}%")

## 8. Store & Category Comparison

In [ ]:
# Sales by store
store_sales = df_agg.groupby('store_id')['units_sold'].agg(['mean', 'sum']).sort_values('sum', ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(store_sales.index, store_sales['sum'], color='#2E86AB', edgecolor='black', alpha=0.8)
plt.xlabel('Store ID', fontsize=12)
plt.ylabel('Total Units Sold', fontsize=12)
plt.title('Total Sales by Store', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Sales by category
cat_sales = df_agg.groupby('cat_id')['units_sold'].agg(['mean', 'sum']).sort_values('sum', ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(cat_sales.index, cat_sales['sum'], color='#A23B72', edgecolor='black', alpha=0.8)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Total Units Sold', fontsize=12)
plt.title('Total Sales by Category', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nCategory Sales Breakdown:")
print(cat_sales)

## 9. Zero Sales Analysis

In [ ]:
# Analyze zero sales
zero_sales_pct = (df_agg['units_sold'] == 0).sum() / len(df_agg) * 100

print(f"Percentage of days with zero sales: {zero_sales_pct:.2f}%")

# Zero sales by store-category
zero_by_group = df_agg.groupby(['store_id', 'cat_id']).apply(
    lambda x: (x['units_sold'] == 0).sum() / len(x) * 100
).reset_index(name='zero_pct')

print("\nZero sales percentage by store-category:")
print(zero_by_group.describe())

## 10. Event Impact Analysis

In [ ]:
# Compare sales on event vs non-event days
event_comparison = df_agg.groupby('has_event')['units_sold'].mean()

plt.figure(figsize=(8, 5))
plt.bar(['No Event', 'Event Day'], event_comparison.values, 
        color=['#2E86AB', '#F18F01'], edgecolor='black', alpha=0.8)
plt.ylabel('Average Units Sold', fontsize=12)
plt.title('Sales Impact of Events', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nNo event average: {event_comparison[0]:.2f}")
print(f"Event day average: {event_comparison[1]:.2f}")
print(f"Event lift: {((event_comparison[1] - event_comparison[0]) / event_comparison[0] * 100):.1f}%")

## 11. Key Insights

### Summary of Findings:

1. **Data Structure**
   - Multiple stores across states
   - Three main product categories
   - Daily sales data spanning multiple years

2. **Seasonality Patterns**
   - Clear weekly seasonality visible
   - Weekend vs weekday differences
   - Event impact on sales

3. **Data Quality**
   - Some zero sales days present
   - Continuous time series after cleaning
   - Ready for forecasting models

4. **Forecasting Approach**
   - Store-category level aggregation reduces noise
   - Weekly seasonality should be captured
   - Calendar features (events, day of week) are important